# Notebook 1 — Introduction to Google Colab and Scientific Data Analysis
## CURE Laboratory: General Chemistry II | HTLV-1 Protease Inhibitor Discovery

**When to complete:** Weeks 2–3  
**Learning objectives:**
- Navigate Google Colab and understand cell types
- Import and inspect a data table (CSV format)
- Calculate mean, standard deviation, and percent error
- Create publication-quality plots with labeled axes, titles, and error bars
- Connect data analysis to General Chemistry II concepts

**Time estimate:** 90–120 minutes (can be split across two sessions)

---
> **How Colab works:**  
> This notebook contains two types of cells: **text cells** (like this one — gray background) and **code cells** (with `[ ]` to the left — white background). Run code cells by clicking the ▶ button or pressing **Shift+Enter**. Read the text cells carefully before running each code cell.

---
## Part 1 — Setting Up Your Environment


### 1.1 Install and import libraries

A **library** (also called a package or module) is a collection of pre-written Python code you can use. Think of it like a toolbox — you don't need to build your own hammer, you just borrow one.

The three libraries we use in this notebook are:
- **NumPy** (`numpy`): Math operations on arrays of numbers — like a scientific calculator that works on whole columns at once
- **Pandas** (`pandas`): Working with tables of data (like Excel, but in Python)
- **Matplotlib** (`matplotlib`): Creating graphs and plots

Run the cell below. If no errors appear, you are ready to go.


In [ ]:
# Run this cell first — always
# It imports the tools we need

import numpy as np          # numerical operations
import pandas as pd         # data tables
import matplotlib.pyplot as plt  # plotting
import matplotlib as mpl

# Make plots look clean and professional
mpl.rcParams['figure.dpi'] = 120
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False

print("Libraries loaded successfully!")
print(f"NumPy version:      {np.__version__}")
print(f"Pandas version:     {pd.__version__}")
print(f"Matplotlib version: {mpl.__version__}")


---
## Part 2 — Creating and Exploring a Data Table

### 2.1 Create sample lab data

In your General Chemistry II lab, you will measure enthalpy changes using calorimetry. The data below represents a typical calorimetry experiment measuring the heat of solution of ammonium chloride (NH₄Cl) in water — an endothermic process you may have studied in lecture.

**Reaction:** NH₄Cl(s) → NH₄⁺(aq) + Cl⁻(aq)    ΔH = +14.8 kJ/mol

The table below shows temperature readings recorded every 30 seconds during an experiment. We will use Python to analyze it exactly as you would analyze real data from your lab notebook.


In [ ]:
# This cell creates your data table
# In future labs, you will replace these numbers with your own measurements

data = {
    'Time_seconds': [0, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300,
                     330, 360, 390, 420, 450, 480, 510, 540, 570, 600],
    'Trial_1_Celsius': [22.1, 22.1, 22.0, 21.8, 21.3, 20.7, 20.2, 19.8,
                        19.6, 19.5, 19.5, 19.6, 19.7, 19.8, 19.9, 20.0,
                        20.1, 20.1, 20.2, 20.2, 20.3],
    'Trial_2_Celsius': [22.3, 22.3, 22.1, 21.9, 21.4, 20.8, 20.3, 19.9,
                        19.7, 19.6, 19.6, 19.7, 19.8, 19.9, 20.0, 20.1,
                        20.1, 20.2, 20.2, 20.3, 20.4],
    'Trial_3_Celsius': [22.0, 22.0, 21.9, 21.7, 21.2, 20.6, 20.1, 19.7,
                        19.5, 19.4, 19.4, 19.5, 19.6, 19.7, 19.8, 19.9,
                        20.0, 20.0, 20.1, 20.1, 20.2]
}

# Create a DataFrame (a table with labeled rows and columns)
df = pd.DataFrame(data)

# Display the first 10 rows
print("First 10 rows of data:")
print(df.head(10))
print(f"\nTotal rows: {len(df)}")
print(f"Columns: {list(df.columns)}")


### 2.2 Calculate statistics for each time point

We need the **mean** (average) temperature across the three trials, plus the **standard deviation** (a measure of how much the trials varied from each other).

Recall from lecture:
- Mean: x̄ = (Σxᵢ) / n
- Standard deviation: s = √[ Σ(xᵢ − x̄)² / (n−1) ]

Python calculates these automatically for each row.

> **Your task:** After running the cell, look at the standard deviation column. At which time point is it largest? At which is it smallest? What does this tell you about measurement consistency?


In [ ]:
# Calculate mean and standard deviation across the three trials
# axis=1 means "calculate across each row" (across trials for each time point)

trial_cols = ['Trial_1_Celsius', 'Trial_2_Celsius', 'Trial_3_Celsius']

df['Mean_Celsius'] = df[trial_cols].mean(axis=1)
df['StdDev_Celsius'] = df[trial_cols].std(axis=1)

# Display the results table
print("Temperature statistics by time point:")
print(df[['Time_seconds', 'Mean_Celsius', 'StdDev_Celsius']].to_string(index=False))

# Find the minimum temperature (the lowest point during dissolution)
min_temp = df['Mean_Celsius'].min()
min_time = df.loc[df['Mean_Celsius'].idxmin(), 'Time_seconds']
initial_temp = df['Mean_Celsius'].iloc[0]
delta_T = min_temp - initial_temp

print(f"\n--- Summary Statistics ---")
print(f"Initial temperature:  {initial_temp:.2f} °C")
print(f"Minimum temperature:  {min_temp:.2f} °C  (at t = {min_time} s)")
print(f"ΔT:                   {delta_T:.2f} °C")


---
## Part 3 — Creating Publication-Quality Plots

### 3.1 Basic temperature vs. time plot

A well-made graph communicates your findings clearly. Every scientific graph must have:
- **Axis labels with units** (never just "x" and "y")
- **A descriptive title**
- **A legend** if multiple data series are shown
- **Appropriate axis ranges** (don't let matplotlib choose random ranges)

The cell below creates a plot showing all three trials plus the mean. Study the code carefully — we will use this same structure for all future notebooks.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# Plot individual trials as thin gray lines
colors = ['#AAAAAA', '#BBBBBB', '#CCCCCC']
labels = ['Trial 1', 'Trial 2', 'Trial 3']
for col, color, label in zip(trial_cols, colors, labels):
    ax.plot(df['Time_seconds'], df[col],
            color=color, linewidth=1.5,
            linestyle='--', alpha=0.7, label=label)

# Plot the mean as a thick colored line with error bars (std dev)
ax.errorbar(df['Time_seconds'], df['Mean_Celsius'],
            yerr=df['StdDev_Celsius'],
            color='#2E5EAA', linewidth=2.5,
            ecolor='#2E5EAA', elinewidth=1, capsize=3,
            label='Mean ± 1 SD', zorder=5)

# Mark the minimum temperature point
min_idx = df['Mean_Celsius'].idxmin()
ax.scatter(df.loc[min_idx, 'Time_seconds'], df.loc[min_idx, 'Mean_Celsius'],
           color='#CC3300', s=100, zorder=6, label='Minimum T')

# Labels and formatting
ax.set_xlabel('Time (seconds)', fontsize=12)
ax.set_ylabel('Temperature (°C)', fontsize=12)
ax.set_title('Dissolution of NH₄Cl in Water: Temperature vs. Time', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('calorimetry_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as calorimetry_plot.png")


### ✏️ ANNOTATION REQUIRED

**In the cell below, write a markdown annotation answering these questions:**
1. Describe what the plot shows (what happened to temperature over time, and why).
2. What chemistry concept explains why the temperature decreased when NH₄Cl dissolved? (Hint: is this reaction endothermic or exothermic? How do you know from the data?)
3. What does the error bar represent, and what does its size tell you about the quality of this data?

*Double-click this text to edit it. Replace the placeholder text with your answers.*


### My Annotation for Plot 1:

*[YOUR ANSWER HERE — describe what you see in the plot and explain the chemistry. Minimum 3–4 sentences.]*


---
## Part 4 — Calculating Enthalpy of Dissolution

### 4.1 From ΔT to ΔH

Now we will use the temperature data to calculate the enthalpy of dissolution. Recall from lecture:

**q = m × c × ΔT**

where:
- q = heat absorbed by the solution (in Joules)
- m = mass of the solution (in grams)
- c = specific heat capacity of water = 4.184 J/(g·°C)
- ΔT = temperature change (T_final − T_initial)

For the **enthalpy of dissolution per mole** of NH₄Cl:

**ΔH_dissolution = −q_solution / n_solute**

(negative sign because heat gained by solution = heat released by dissolving process, and vice versa)


In [ ]:
# Experimental conditions
mass_solution_g = 100.0   # mass of water used (grams)
mass_NH4Cl_g = 5.35        # mass of NH4Cl dissolved (grams)
MW_NH4Cl = 53.49           # molar mass of NH4Cl (g/mol)
c_water = 4.184            # specific heat of water (J/g·°C)

# Calculate moles of NH4Cl
n_NH4Cl = mass_NH4Cl_g / MW_NH4Cl
print(f"Moles of NH4Cl: {n_NH4Cl:.4f} mol")

# Use mean temperature data
T_initial = df['Mean_Celsius'].iloc[0]
T_final = df['Mean_Celsius'].min()       # minimum temperature = maximum cooling
delta_T = T_final - T_initial
print(f"ΔT (solution): {delta_T:.3f} °C")

# Calculate heat absorbed by the solution
q_solution = mass_solution_g * c_water * delta_T   # will be negative (cooling)
print(f"q_solution: {q_solution:.2f} J")

# Calculate molar enthalpy of dissolution
# q_solution is negative because solution cooled → endothermic dissolution absorbs heat from solution
delta_H_J_per_mol = -q_solution / n_NH4Cl
delta_H_kJ_per_mol = delta_H_J_per_mol / 1000
print(f"\nΔH_dissolution (experimental): +{delta_H_kJ_per_mol:.2f} kJ/mol")
print(f"ΔH_dissolution (literature):   +14.8 kJ/mol")

# Calculate percent error
percent_error = abs(delta_H_kJ_per_mol - 14.8) / 14.8 * 100
print(f"Percent error: {percent_error:.1f}%")


### ✏️ ANNOTATION REQUIRED

Answer in the markdown cell below:
1. Is the percent error acceptable for a calorimetry experiment? What sources of error might explain the difference from the literature value?
2. The sign of ΔH is positive (+). What does this mean in terms of heat flow and bond breaking/forming? How does this connect to what you observed in the temperature data?
3. Why do we take the **minimum** temperature (rather than the final temperature) as T_final in this calculation?


### My Annotation for Section 4:

*[YOUR ANSWER HERE — address all three questions above. Minimum 4–5 sentences.]*


---
## Part 5 — Comparing Multiple Experiments: Bar Charts

### 5.1 Comparing ΔH across different solutes

Imagine you ran the same calorimetry experiment with three different ionic compounds. The table below shows results from three different lab sections. We will create a bar chart comparing the results.

This type of comparison is exactly what your team will do in Phase 3 of the CURE — comparing results across three different inhibitor systems. Bar charts with error bars are one of the most common ways to display comparative data in chemistry research papers.


In [ ]:
# Results from three lab sections measuring different ionic compounds
compounds = ['NH₄Cl', 'NaNO₃', 'KBr']
delta_H_values = [15.2, 20.5, 19.9]     # kJ/mol (experimental)
delta_H_lit = [14.8, 20.5, 19.9]        # kJ/mol (literature)
std_devs = [0.8, 1.2, 0.6]             # standard deviation from multiple trials

fig, ax = plt.subplots(figsize=(8, 5))

x = np.arange(len(compounds))
width = 0.35

# Experimental bars
bars1 = ax.bar(x - width/2, delta_H_values, width,
               label='Experimental', color='#2E5EAA', alpha=0.85,
               yerr=std_devs, capsize=5, error_kw={'elinewidth': 1.5})

# Literature bars
bars2 = ax.bar(x + width/2, delta_H_lit, width,
               label='Literature', color='#CC6600', alpha=0.85,
               capsize=5)

# Labels
ax.set_xlabel('Ionic Compound', fontsize=12)
ax.set_ylabel('ΔH of Dissolution (kJ/mol)', fontsize=12)
ax.set_title('Enthalpy of Dissolution: Experimental vs. Literature Values', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(compounds, fontsize=12)
ax.legend(fontsize=10)
ax.axhline(0, color='black', linewidth=0.8)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('enthalpy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as enthalpy_comparison.png")


### ✏️ ANNOTATION REQUIRED

Answer in the markdown cell below:
1. Which compound has the largest discrepancy between experimental and literature values? What might cause this?
2. What does the error bar represent in this context, and how does its size compare across compounds?
3. **Connection to CURE:** In Phase 3, you will compare RMSD values (or H-bond occupancy values) across three inhibitor systems. How is that comparison structurally similar to the comparison you just made? What type of plot would be appropriate?


### My Annotation for Section 5:

*[YOUR ANSWER HERE]*


---
## ✅ Notebook 1 Completion Checklist

Before submitting this notebook, confirm that:

- [ ] All code cells have been run (no empty `[ ]` to the left — they should show `[1]`, `[2]`, etc.)
- [ ] All three **ANNOTATION REQUIRED** sections are filled in with complete answers (not placeholder text)
- [ ] Both plots display correctly (temperature vs. time, and bar chart)
- [ ] You have saved the notebook (File → Save, or Ctrl+S)

**To submit:** Download as .ipynb (File → Download → Download .ipynb) and upload to the course LMS.

---

## 🔗 Connection to Future Notebooks

In Notebook 2, you will work with molecular coordinate data (PDB files) from protein structures. In Notebooks 3–5, you will apply the same statistical and plotting skills you practiced here — but instead of temperature data, you will be analyzing **RMSD values** and **hydrogen bond counts** from molecular dynamics simulations. The code structure is nearly identical; only the data changes.
